In [1]:
# ONCOPLOT MATRIX DEVELOPMENT

In [2]:
print("ONCOPLOT MATRIX")
print("%" * 50)

import pandas as pd
import numpy as np

ONCOPLOT MATRIX
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


In [20]:
# Load cleaned data
mutations = pd.read_csv("../data/clean_mutations.tsv", sep="\t")
clinical = pd.read_csv("../data/clean_clinical.tsv", sep="\t")

print(f"Loaded: {mutations.shape} mutations, {clinical.shape} clinical")
print("\nColumns:", list(mutations.columns[:10]))

Loaded: (84690, 12) mutations, (528, 37) clinical

Columns: ['Hugo_Symbol', 'Entrez_Gene_Id', 'Variant_Classification', 'Tumor_Sample_Barcode', 'Chromosome', 'Start_Position', 'End_Position', 'HGVSp_Short', 'SAMPLE_ID', 'hypermutator']


In [27]:
non_hyper_mut = mutations[mutations['hypermutator'] != 1]
total_samples = non_hyper_mut['SAMPLE_ID'].nunique()  # 504
print(f"\nNon-hypermutated samples: {total_samples}")

# Top 20 GENES
print("\n TOP 20 GENES")
gene_freq = (
    non_hyper_mut.groupby('Hugo_Symbol')['SAMPLE_ID']
                 .nunique()
                 .transform(lambda x: x / total_samples * 100)
                 .sort_values(ascending=False)
)

top_genes = gene_freq.head(20).index.tolist()
print(gene_freq.head(20).round(1))
print(f"\nTop genes: {top_genes}")


Non-hypermutated samples: 504

 TOP 20 GENES
Hugo_Symbol
TP53       70.6
TTN        41.7
FAT1       22.6
CDKN2A     21.8
FRG1B      20.0
CSMD3      19.8
MUC16      18.8
PIK3CA     18.5
NOTCH1     17.9
SYNE1      17.5
LRP1B      16.7
KMT2D      16.1
PCLO       15.9
FLG        14.1
DNAH5      13.1
USH2A      12.1
NSD1       11.5
RYR2       11.1
PKHD1L1    10.9
CASP8      10.3
Name: SAMPLE_ID, dtype: float64

Top genes: ['TP53', 'TTN', 'FAT1', 'CDKN2A', 'FRG1B', 'CSMD3', 'MUC16', 'PIK3CA', 'NOTCH1', 'SYNE1', 'LRP1B', 'KMT2D', 'PCLO', 'FLG', 'DNAH5', 'USH2A', 'NSD1', 'RYR2', 'PKHD1L1', 'CASP8']


In [22]:
# FILTER + SEVERITY RANKING
print("\n FILTER TOP GENES + SEVERITY...")
mut_top = mutations[
    (mutations['Hugo_Symbol'].isin(top_genes)) & (mutations['hypermutator'] !=1)].copy()


 FILTER TOP GENES + SEVERITY...


In [23]:
# Severity priority (truncating > missense)
severity = {
    'Nonsense_Mutation': 9, 'Frame_Shift_Del': 8, 'Frame_Shift_Ins': 8,
    'Splice_Site': 7, 'Nonstop_Mutation': 7, 'Translation_Start_Site': 6, 'In_Frame_Del': 5, 'In_Frame_Ins': 5, 'Missense_Mutation': 4
}

mut_top['severity'] = mut_top['Variant_Classification'].map(severity).fillna(0)

print(f"Top genes mutations: {len(mut_top):,}")
print(f"Severity distribution:")
print(mut_top['severity'].value_counts().sort_index())

Top genes mutations: 2,789
Severity distribution:
severity
4    1936
5      29
6       1
7     119
8     256
9     448
Name: count, dtype: int64


In [24]:
# Keep most severe mutation per gene-sample
print("\nMost severe per gene-sample...")
matrix_long = mut_top.loc[
    mut_top.groupby(['Hugo_Symbol', 'SAMPLE_ID'])['severity'].idxmax()]
print(f"Multi-hit: {len(mut_top):,} -> {len(matrix_long):,} unique")


Most severe per gene-sample...
Multi-hit: 2,789 -> 2,021 unique


In [25]:
# PIVOT TO MATRIX
print("\n PIVOT MATRIX...")
oncoplot_matrix = matrix_long.pivot_table(
    index='Hugo_Symbol', columns='SAMPLE_ID', values='Variant_Classification', aggfunc='first'
).reindex(index=top_genes)

print(f"Matrix shape: {oncoplot_matrix.shape}")
print("\nMatrix preview (first 5 genes, 10 samples):")
print(oncoplot_matrix.iloc[:5, :10].fillna(''))


 PIVOT MATRIX...
Matrix shape: (20, 482)

Matrix preview (first 5 genes, 10 samples):
SAMPLE_ID      TCGA-4P-AA8J-01    TCGA-BA-4074-01    TCGA-BA-4075-01  \
Hugo_Symbol                                                            
TP53         Nonsense_Mutation  Missense_Mutation        Splice_Site   
TTN                                                Nonsense_Mutation   
FAT1           Frame_Shift_Ins                       Frame_Shift_Ins   
CDKN2A                                                   Splice_Site   
FRG1B                                                                  

SAMPLE_ID      TCGA-BA-4076-01    TCGA-BA-4077-01    TCGA-BA-4078-01  \
Hugo_Symbol                                                            
TP53           Frame_Shift_Ins                           Splice_Site   
TTN          Nonsense_Mutation  Missense_Mutation  Missense_Mutation   
FAT1                                               Missense_Mutation   
CDKN2A                                          

In [29]:
# SORT + SAVE

# Binary matrix for sorting (mutated=1, wildtype=0)
binary_matrix = oncoplot_matrix.notna().astype(int)


# Sort by top 5 genes (TP53, TTN/FAT1, etc.)
sort_genes = top_genes[:5]
print("Sorting by:", sort_genes)


sorted_samples = (
    binary_matrix.loc[sort_genes]
        .T
        .sort_values(by=sort_genes, ascending=False)
        .index
)

oncoplot_matrix_sorted = oncoplot_matrix[sorted_samples]


print(f"Sorted matrix: {oncoplot_matrix_sorted.shape}")
print("\nFirst 5 samples - TP53 status:")
print(oncoplot_matrix_sorted.iloc[:5, :5].fillna('WT'))
print("\nTP53 mutated %:", (binary_matrix.loc['TP53'].sum() / len(binary_matrix.columns) * 100).round(1), "%")


# SAVE
oncoplot_matrix_sorted.to_csv('../data/oncoplot_matrix.csv')
print("\n SAVED: data/oncoplot_matrix.csv")

Sorting by: ['TP53', 'TTN', 'FAT1', 'CDKN2A', 'FRG1B']
Sorted matrix: (20, 482)

First 5 samples - TP53 status:
SAMPLE_ID      TCGA-CN-5370-01    TCGA-CR-7379-01    TCGA-UF-A7J9-01  \
Hugo_Symbol                                                            
TP53               Splice_Site  Missense_Mutation  Missense_Mutation   
TTN          Missense_Mutation  Missense_Mutation  Missense_Mutation   
FAT1           Frame_Shift_Del  Missense_Mutation  Nonsense_Mutation   
CDKN2A       Nonsense_Mutation        Splice_Site    Frame_Shift_Del   
FRG1B        Missense_Mutation  Missense_Mutation  Missense_Mutation   

SAMPLE_ID      TCGA-BA-4075-01    TCGA-BA-4078-01  
Hugo_Symbol                                        
TP53               Splice_Site        Splice_Site  
TTN          Nonsense_Mutation  Missense_Mutation  
FAT1           Frame_Shift_Ins  Missense_Mutation  
CDKN2A             Splice_Site    Frame_Shift_Del  
FRG1B                       WT                 WT  

TP53 mutated %: 73